0) Mount Drive + paths

In [1]:
from google.colab import drive
drive.mount('/content/drive')

ZIP_PATH = "/content/drive/MyDrive/TFM/datasets/weapons_cs231n/raw_zip/weapons_yolov8.zip"

# Carpeta de trabajo en Colab (se borra al reiniciar runtime)
WORK_DIR = "/content/weapons_det"
DATA_DIR = f"{WORK_DIR}/dataset_full"
MINI_DIR = f"{WORK_DIR}/dataset_mini"

# Donde guardar resultados persistentes
PROJECT_DIR = "/content/drive/MyDrive/TFM/experiments/weapon_det"

Mounted at /content/drive


1) Copiar zip a /content y descomprimir ahí

In [3]:
import os, shutil, zipfile

os.makedirs(WORK_DIR, exist_ok=True)
local_zip = f"{WORK_DIR}/dataset.zip"

print("Copiando zip a /content ...")
shutil.copy(ZIP_PATH, local_zip)

print("Descomprimiendo ...")
os.makedirs(DATA_DIR, exist_ok=True)
with zipfile.ZipFile(local_zip, 'r') as z:
    z.extractall(DATA_DIR)

print("OK. Contenido extraído en:", DATA_DIR)

Copiando zip a /content ...
Descomprimiendo ...
OK. Contenido extraído en: /content/weapons_det/dataset_full


2) Detectar estructura y localizar data.yaml

In [4]:
import glob, os

# Buscar data.yaml en cualquier subcarpeta
candidates = glob.glob(f"{DATA_DIR}/**/data.yaml", recursive=True) + glob.glob(f"{DATA_DIR}/**/*.yaml", recursive=True)
print("YAML candidates:", candidates[:5], "..." if len(candidates) > 5 else "")

if not candidates:
    raise FileNotFoundError("No encuentro data.yaml dentro del zip descomprimido. Revisa la estructura.")

# Prioriza el que se llame exactamente data.yaml
yaml_path = None
for c in candidates:
    if os.path.basename(c) == "data.yaml":
        yaml_path = c
        break
yaml_path = yaml_path or candidates[0]

ROOT = os.path.dirname(yaml_path)
print("Usaré ROOT =", ROOT)
print("data.yaml =", yaml_path)

YAML candidates: ['/content/weapons_det/dataset_full/data.yaml', '/content/weapons_det/dataset_full/data.yaml'] 
Usaré ROOT = /content/weapons_det/dataset_full
data.yaml = /content/weapons_det/dataset_full/data.yaml


Chequeo rápido de estructura (images/labels y splits)

In [5]:
import os, glob

def count_files(pat):
    return len(glob.glob(pat, recursive=True))

# Posibles nombres de split según export (train/valid/test o train/val/test)
splits = ["train", "valid", "val", "test"]

for s in splits:
    img_dir = os.path.join(ROOT, s, "images")
    lbl_dir = os.path.join(ROOT, s, "labels")
    if os.path.isdir(img_dir) or os.path.isdir(lbl_dir):
        n_imgs = count_files(os.path.join(img_dir, "*")) if os.path.isdir(img_dir) else 0
        n_lbls = count_files(os.path.join(lbl_dir, "*")) if os.path.isdir(lbl_dir) else 0
        print(f"{s:>5} | images: {n_imgs:>6} | labels: {n_lbls:>6} | img_dir exists: {os.path.isdir(img_dir)} | lbl_dir exists: {os.path.isdir(lbl_dir)}")

train | images:  14804 | labels:  14804 | img_dir exists: True | lbl_dir exists: True
valid | images:   2544 | labels:   2544 | img_dir exists: True | lbl_dir exists: True
 test | images:    623 | labels:    623 | img_dir exists: True | lbl_dir exists: True


4) Validación de emparejado imagen-label

In [6]:
import os, glob

IMG_EXTS = (".jpg", ".jpeg", ".png", ".bmp", ".webp")

def list_stems(folder, exts=None):
    files = glob.glob(os.path.join(folder, "*"))
    stems = set()
    for f in files:
        base = os.path.basename(f)
        stem, ext = os.path.splitext(base)
        if exts is None or ext.lower() in exts:
            stems.add(stem)
    return stems

def check_split(split_name):
    img_dir = os.path.join(ROOT, split_name, "images")
    lbl_dir = os.path.join(ROOT, split_name, "labels")
    if not (os.path.isdir(img_dir) and os.path.isdir(lbl_dir)):
        return

    img_stems = list_stems(img_dir, IMG_EXTS)
    lbl_stems = list_stems(lbl_dir, None)  # labels suelen ser .txt
    missing_lbl = sorted(list(img_stems - lbl_stems))[:10]
    missing_img = sorted(list(lbl_stems - img_stems))[:10]

    print(f"\n[{split_name}]")
    print("  imgs:", len(img_stems), "labels:", len(lbl_stems))
    print("  imgs sin label (muestra):", missing_lbl if missing_lbl else "OK")
    print("  labels sin img (muestra):", missing_img if missing_img else "OK")

for s in ["train", "valid", "val", "test"]:
    check_split(s)


[train]
  imgs: 14804 labels: 14804
  imgs sin label (muestra): OK
  labels sin img (muestra): OK

[valid]
  imgs: 2544 labels: 2544
  imgs sin label (muestra): OK
  labels sin img (muestra): OK

[test]
  imgs: 623 labels: 623
  imgs sin label (muestra): OK
  labels sin img (muestra): OK


5) Crear mini-dataset (submuestreo)

In [7]:
import os, random, shutil, glob, yaml

random.seed(42)

# Ajusta estos tamaños según quieras (pequeño pero suficiente para smoke test)
N_TRAIN = 200
N_VAL   = 80

# Detecta si el split es "valid" o "val"
VAL_SPLIT = "valid" if os.path.isdir(os.path.join(ROOT, "valid")) else "val"
assert os.path.isdir(os.path.join(ROOT, "train")), "No existe split train en ROOT"
assert os.path.isdir(os.path.join(ROOT, VAL_SPLIT)), f"No existe split {VAL_SPLIT} en ROOT"

def sample_and_copy(split, n):
    src_img = os.path.join(ROOT, split, "images")
    src_lbl = os.path.join(ROOT, split, "labels")
    dst_img = os.path.join(MINI_DIR, split, "images")
    dst_lbl = os.path.join(MINI_DIR, split, "labels")
    os.makedirs(dst_img, exist_ok=True)
    os.makedirs(dst_lbl, exist_ok=True)

    imgs = [p for p in glob.glob(os.path.join(src_img, "*")) if os.path.splitext(p)[1].lower() in IMG_EXTS]
    if len(imgs) == 0:
        raise RuntimeError(f"No encuentro imágenes en {src_img}")

    chosen = random.sample(imgs, k=min(n, len(imgs)))

    copied = 0
    for img_path in chosen:
        stem = os.path.splitext(os.path.basename(img_path))[0]
        lbl_path = os.path.join(src_lbl, stem + ".txt")
        if not os.path.exists(lbl_path):
            # si falta label, saltamos para no romper entrenamiento
            continue
        shutil.copy(img_path, os.path.join(dst_img, os.path.basename(img_path)))
        shutil.copy(lbl_path, os.path.join(dst_lbl, os.path.basename(lbl_path)))
        copied += 1

    print(f"{split}: seleccionadas {len(chosen)}, copiadas con label {copied}")
    return copied

# Re-crear MINI_DIR desde cero
if os.path.exists(MINI_DIR):
    shutil.rmtree(MINI_DIR)
os.makedirs(MINI_DIR, exist_ok=True)

c_train = sample_and_copy("train", N_TRAIN)
c_val = sample_and_copy(VAL_SPLIT, N_VAL)

print("Mini dataset creado en:", MINI_DIR)

train: seleccionadas 200, copiadas con label 200
valid: seleccionadas 80, copiadas con label 80
Mini dataset creado en: /content/weapons_det/dataset_mini


6) Crear data.yaml para el mini-dataset

In [8]:
import os, yaml

with open(yaml_path, "r") as f:
    original_yaml = yaml.safe_load(f)

names = original_yaml.get("names", None)
nc = original_yaml.get("nc", None)

# Si names viene como lista o dict, lo normalizamos
if isinstance(names, dict):
    # dict {0:"gun"} -> lista ["gun"]
    names_list = [names[k] for k in sorted(names.keys())]
elif isinstance(names, list):
    names_list = names
else:
    # fallback: clase única si no está
    names_list = ["weapon"]

mini_yaml = {
    "path": MINI_DIR,
    "train": "train/images",
    "val": f"{'valid' if VAL_SPLIT=='valid' else 'val'}/images",
    "names": {i: n for i, n in enumerate(names_list)}
}

mini_yaml_path = os.path.join(MINI_DIR, "data.yaml")
with open(mini_yaml_path, "w") as f:
    yaml.safe_dump(mini_yaml, f, sort_keys=False, allow_unicode=True)

print("data.yaml mini creado:", mini_yaml_path)
print(mini_yaml)

data.yaml mini creado: /content/weapons_det/dataset_mini/data.yaml
{'path': '/content/weapons_det/dataset_mini', 'train': 'train/images', 'val': 'valid/images', 'names': {0: 'gun'}}


7) Instalar Ultralytics y hacer smoke test (2–3 epochs)

In [9]:
!pip -q install ultralytics

from ultralytics import YOLO

# Modelo pequeño para prueba rápida
model = YOLO("yolov8n.pt")

results = model.train(
    data=mini_yaml_path,
    epochs=3,
    imgsz=640,
    batch=16,
    workers=2,
    project=PROJECT_DIR,
    name="smoke_weapons_yolov8n_mini",
    exist_ok=True
)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 20.7 MB/s eta 0:00:00
Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Ultralytics 8.4.15 🚀 Python-3.12.12 torch-2.10.0+cpu CPU (Intel Xeon CPU @ 2.20GHz)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/weapons_det/dataset_mini/data.yaml, degrees=0.0, deterministic=True, device=cpu, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=3, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript

8) Inferencia rápida en 5 imágenes del mini-val (para ver que “detecta algo”)

In [10]:
import glob, random
from ultralytics import YOLO

best = f"{PROJECT_DIR}/smoke_weapons_yolov8n_mini/weights/best.pt"
model2 = YOLO(best)

val_img_dir = f"{MINI_DIR}/{ 'valid' if VAL_SPLIT=='valid' else 'val' }/images"
samples = glob.glob(val_img_dir + "/*")
samples = random.sample(samples, k=min(5, len(samples)))

pred = model2.predict(source=samples, imgsz=640, conf=0.25)


0: 640x640 (no detections), 332.5ms
1: 640x640 (no detections), 332.5ms
2: 640x640 (no detections), 332.5ms
3: 640x640 (no detections), 332.5ms
4: 640x640 (no detections), 332.5ms
Speed: 5.7ms preprocess, 332.5ms inference, 0.7ms postprocess per image at shape (1, 3, 640, 640)
